## Решение дает 12 баллов из 15 при отправке в систему проверки stepik.



In [ ]:
!pip install openai langchain tiktoken langchain-openai langchain-community sentence_transformers faiss-cpu -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.3/268.3 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 817.7/817.7 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 46.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.3/163.3 kB 13.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 28.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.5/287.5 kB 14.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.0/113.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 926.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!wget https://raw.githubusercontent.com/a-milenkin/LLM_practical_course/main/notebooks/utils.py

--2024-04-15 15:41:31--  https://raw.githubusercontent.com/a-milenkin/LLM_practical_course/main/notebooks/utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10823 (11K) [text/plain]
Saving to: ‘utils.py’

utils.py            100%[===================>]  10.57K  --.-KB/s    in 0s      

2024-04-15 15:41:31 (65.0 MB/s) - ‘utils.py’ saved [10823/10823]



In [ ]:
from utils import ChatOpenAI
from getpass import getpass

#course_api_key= "Введите ваш API ключ с курса"
course_api_key = getpass(prompt='API key')

# Инициализируем языковую модель
llm = ChatOpenAI(temperature=0.0, course_api_key=course_api_key)

API key··········


Загружаем файлы

In [ ]:
!wget https://stepik.org/media/attachments/lesson/1084288/The_Daughter_of_The_Commandant.pdf

--2024-04-15 15:46:10--  https://stepik.org/media/attachments/lesson/1084288/The_Daughter_of_The_Commandant.pdf
Resolving stepik.org (stepik.org)... 178.248.239.111
Connecting to stepik.org (stepik.org)|178.248.239.111|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 198595 (194K) [application/pdf]
Saving to: ‘The_Daughter_of_The_Commandant.pdf’

The_Daughter_of_The 100%[===================>] 193.94K   444KB/s    in 0.4s    

2024-04-15 15:46:12 (444 KB/s) - ‘The_Daughter_of_The_Commandant.pdf’ saved [198595/198595]



In [ ]:
import pandas as pd
from tqdm import tqdm

df = pd.read_csv('https://stepik.org/media/attachments/lesson/1084288/pushkin_questions.csv')

Соберем все компоненты для RAG


<center> <img src='https://github.com/a-milenkin/LLM_practical_course/blob/main/images/RAG.png?raw=1' width="800" height="250">


In [ ]:
!pip install pypdf -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 2.6 MB/s eta 0:00:00


In [ ]:
# Определите Document loader для pdf
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/The_Daughter_of_The_Commandant.pdf")
docs = loader.load() # метод load сразу разбивает текст на документы по страницам

Следующие две ячейки можно пропустить, если нас устраивает сплит в методе load

In [ ]:
docs = [docs[i].page_content for i in range(len(docs))] # собираем текст, если хотим использовать свой сплитер

In [ ]:
# Определите подходящий Splitter
from langchain.text_splitter import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)

splitter = CharacterTextSplitter(
    separator="\n\n",  # символ-разделитель, по умолчанию переход к новому абзацу '\n\n'
    chunk_size=500,  # размер документа в символах
    chunk_overlap=100,  # насколько соседние документы могут перекрывать друг-друга
    length_function=len,  # функция, по которой считается размер документа
    is_separator_regex=False,  # является ли разделитель регулярным выражением
)


split_documents = splitter.create_documents(docs)
len(split_documents)

4

# Выбираем Embedding models

## 🔱 Эмбеддинги от OpenAI (API)

In [ ]:
# Если используете ключ курса, запустите эту ячейку
from utils import OpenAIEmbeddings

embeddings_api_model = OpenAIEmbeddings(course_api_key=course_api_key)

# Выбираем 🗂 Vector Store

In [ ]:
from langchain.vectorstores import FAISS

embeddings = OpenAIEmbeddings(course_api_key=course_api_key)
db = FAISS.from_documents(
    split_documents[:4], embeddings
)

db.save_local("faiss_db")  # можно сохранить базу локально, указав путь

# Определим 🎣 Retriever

In [ ]:
# Самый частый кейс - использование векторного хранилища и его методов для получения документов
retriever = db.as_retriever(
    search_type="similarity",  # тип поиска похожих документов
    k=4,  # количество возвращаемых документов (Default: 4)
    score_threshold=None,  # минимальный порог для поиска "similarity_score_threshold"
)

# 🚰 RAG Pipeline - подключаем RAG к LLM

In [ ]:
from langchain.schema import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Создаём простой шаблон
template = """
Answer the question based only on the following context:

{context}

Question: {question}
"""
# Создаём промпт из шаблона
prompt = ChatPromptTemplate.from_template(template)


# Объявляем функцию, которая будет собирать строку из полученных документов
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])


# Создаём цепочку
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
answers = [] # Список, где будем хранить ответы модели

for text_input in tqdm(df['question']):
    answer = chain.invoke(text_input)
    answers.append(answer) # Добавляем ответ в список
    break # Для отладки. Уберите, когда убедитесь, что на одном примере работает

  0%|          | 0/15 [00:02<?, ?it/s]


In [ ]:
df['answer'] = answers # Создаём новый столбец из ответов модели

In [ ]:
df.to_csv('4_1_11_solution.csv', index=False) # Сохраняем файл, отправляем на Stepik, получаем баллы :)